# 📓 02 — 自定义 sector weights 跑归因

**目标**: 学会用 `attribute_index()` 跑归因,并用自己的 weights 验证。

**适合**: 想做"如果我调整权重,归因结果怎么变"分析的用户。

---

**默认 weights** 在 `config/sector_weights.json` 是 2026-Q2 近似值。
本 notebook 演示怎么**临时覆盖**做 what-if 分析。

In [ ]:
# 第 0 步: import (含 robust path 修复)
import sys
from pathlib import Path

def _find_project_root():
    cwd = Path.cwd()
    for cand in [cwd, *cwd.parents]:
        if (cand / 'src').is_dir() and (cand / 'config').is_dir():
            return cand
    return cwd

PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

import json
from src.attribution import attribute_index, load_sector_weights, get_sector_returns

In [ ]:
# 第 1 步: 看默认 weights (DIA / QQQ / RSP / QQQE × 11 GICS)
# 注意: weights dict 里有 "note" (str) 字段,需要过滤
default_w = load_sector_weights()
for sym, weights in default_w.items():
    if sym.startswith('_'):  # 跳过 _meta
        continue
    print(f"\n=== {sym} ===")
    # 只取数字 (sector weights),跳过 "note" 字符串
    sector_weights = [(k, v) for k, v in weights.items() if isinstance(v, (int, float))]
    sorted_w = sorted(sector_weights, key=lambda x: -x[1])
    for sector, w in sorted_w[:5]:  # top 5
        print(f"  {sector:25s} {w*100:>5.1f}%")

In [ ]:
# 第 2 步: 用默认 weights 跑 QQQ 5 日归因
r_default = attribute_index('QQQ', lookback_days=5)
print("=== QQQ 5 日归因 (默认 weights) ===")
print(f"  实际:    {r_default['actual_return_pct']:+.2f}%")
print(f"  预测:    {r_default['predicted_return_pct']:+.2f}%")
print(f"  残差:    {r_default['residual_pct']:+.2f}%")
print(f"  Top 3 主升: ")
sorted_c = sorted(r_default['sector_contributions_pct'].items(), key=lambda x: -x[1])
for s, v in sorted_c[:3]:
    print(f"    {s:25s} {v:+.2f}%")

In [ ]:
# 第 3 步: 改 weights — QQQ 科技集中度假设 (XLK 40% → 50%)
# 直接复算: sum(w * sector_return) - 不调 attribute_index (它读 config)
# 注意: default_w['QQQ'] 里有 "note" 字符串,过滤掉
qqq_default = {k: v for k, v in default_w['QQQ'].items() if isinstance(v, (int, float))}
modified_w = qqq_default.copy()
modified_w['XLK'] = 0.50  # 从默认 52% 提到 50%(看跟默认差不多)
modified_w['XLY'] = qqq_default.get('XLY', 0.13) * 0.5  # 削减消费

print("Modified QQQ weights:")
for s, w in sorted(modified_w.items(), key=lambda x: -x[1])[:5]:
    print(f"  {s:25s} {w*100:>5.1f}%")

In [ ]:
# 第 4 步: 用 modified weights 复算归因
import pandas as pd
from src.returns import compute_returns

sector_rets = get_sector_returns('2024-07-01', '2026-07-10')
qqq_returns = sector_rets['QQQ']
sector_only = sector_rets.drop(columns=['DIA', 'QQQ', 'RSP', 'QQQE'])

# 5 日窗口
last_5d = sector_only.tail(5).sum()  # log returns 可加
actual_5d = qqq_returns.tail(5).sum() * 100

predicted_default = sum(
    qqq_default.get(s, 0) * last_5d[s] for s in sector_only.columns
) * 100
predicted_modified = sum(
    modified_w.get(s, 0) * last_5d[s] for s in sector_only.columns
) * 100

print(f"=== QQQ 5 日归因 (XLK 50% what-if) ===")
print(f"  实际:        {actual_5d:+.2f}%")
print(f"  预测 (默认): {predicted_default:+.2f}%, 残差 {actual_5d - predicted_default:+.2f}%")
print(f"  预测 (XLK 50%): {predicted_modified:+.2f}%, 残差 {actual_5d - predicted_modified:+.2f}%")
print(f"\n💡 改 weight 后,预测变化 {predicted_modified - predicted_default:+.2f}%")
print(f"   这就是 what-if 分析: weight 改了,模型认为 QQQ 应该多涨/少涨多少")

## 💡 为什么 manual 复算

`attribute_index()` 当前**不接 weights_override 参数**。
v0.5.0 Phase 4 完成后,下一个版本会加 `weights_override`,这样一行就能跑 what-if。

**目前 workaround** (上面 cell 演示):
1. 调 `get_sector_returns()` 拿 sector log returns
2. 自己用 `sum(w * r)` 算 weighted return
3. 对比默认 weights vs 修改 weights 的预测差

**结果**: 不调 attribute.py,纯用公开函数做 what-if。

In [ ]:
# 第 5 步: 多指数批量归因对比
from src.attribution import attribute_all_indices

all_r = attribute_all_indices(lookback_days=5)
print(f"{'index':6s} {'actual':>8s} {'predicted':>10s} {'residual':>9s} {'verdict':>12s}")
print("-" * 50)
for r in all_r:
    abs_res = abs(r['residual_pct'])
    verdict = 'ok' if abs_res < 0.5 else ('watch' if abs_res < 1.0 else 'large')
    print(f"{r['index']:6s} {r['actual_return_pct']:>+7.2f}% "
          f"{r['predicted_return_pct']:>+9.2f}% "
          f"{r['residual_pct']:>+8.2f}% {verdict:>12s}")

## 🎯 练习

1. **改 1 个 sector weight 跑 what-if**: 把 QQQ 的 XLK 从 40% → 35%,XLF 从 8% → 13%
2. **加新 sector**: 如果要加 'semiconductor' 行业,怎么加?
3. **跟 RSP 等权对比**: RSP 是等权,所有 sector 都是 1/11 = 9.09%。看残差对比